# W11: 소매 종합 3패널 대시보드

매출/주문/재고 회전율 지표를 한 화면에서 확인합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

orders = pd.DataFrame({
    "날짜": pd.date_range("2026-01-01", periods=14, freq="D").astype(str),
    "매출": [120000, 98000, 132000, 145000, 151000, 137000, 142000, 149000, 160000, 168000, 162000, 171000, 180000, 175000],
    "주문수": [80, 75, 92, 105, 110, 98, 102, 115, 121, 130, 128, 134, 145, 149],
    "재고회전율": [1.2, 1.1, 1.4, 1.5, 1.6, 1.3, 1.4, 1.7, 1.8, 1.9, 2.0, 2.2, 2.1, 2.3]
})

# 3-panel dashboard
fig, axs = plt.subplots(1, 3, figsize=(14, 3))
axs[0].plot(orders["날짜"], orders["매출"], color="#4c72b0")
axs[0].set_title("매출 추이")
axs[1].plot(orders["날짜"], orders["주문수"], color="#55a868")
axs[1].set_title("주문 수")
axs[2].plot(orders["날짜"], orders["재고회전율"], color="#c44e52")
axs[2].set_title("재고 회전율")
plt.setp(axs[0].get_xticklabels(), rotation=30)
plt.setp(axs[1].get_xticklabels(), rotation=30)
plt.setp(axs[2].get_xticklabels(), rotation=30)
plt.tight_layout()
plt.show()

comment = use_gemini(
    "매출/주문/재고회전율 수치를 3문장으로 운영 인사이트로 요약해줘.",
    "매출과 주문수가 동반 상승하며 회전율도 개선되고 있습니다. 판매 집중일을 분석해 재고 안전 재조정 필요합니다."
)
print(comment)
